# Notebook 02 — Funciones de strings

Segundo sub-bloque del Tema 05. Las funciones de string son las herramientas para **limpiar, extraer y transformar texto** directamente desde SQL — sin pasar por Python.

En este notebook cubres capitalización (`upper`, `lower`, `initcap`), limpieza (`trim` y variantes), búsqueda (`position`, `LIKE`, `ILIKE`), extracción (`substring`, `split_part`, `left`, `right`), composición (`||`, `concat`, `format`) y un primer encuentro con **regex** vía `regexp_replace` y `regexp_match`.

El caso pedagógico fuerte cierra el notebook: parsear las columnas sucias de Airbnb (`price`, `bathrooms_text`, `host_response_rate`) usando todo lo aprendido.

## Setup

Re-cargar la extensión `sql` y reconectar el engine — notebook auto-contenido.

In [ ]:
%load_ext sql

from sqlalchemy import create_engine

AURORA_HOST     = "aurora-mod4.cluster-xxxxx.us-east-1.rds.amazonaws.com"
AURORA_PASSWORD = "TU_PASSWORD_AQUI"
AURORA_DATABASE = "northwind"

engine = create_engine(
    f"postgresql+psycopg2://postgres:{AURORA_PASSWORD}@{AURORA_HOST}:5432/{AURORA_DATABASE}"
)

%sql engine

## Capitalización — `upper`, `lower`, `initcap`

Tres funciones simples, una para cada formato:

| Función | `'hola mundo'` → |
|---|---|
| `UPPER(s)` | `'HOLA MUNDO'` |
| `LOWER(s)` | `'hola mundo'` |
| `INITCAP(s)` | `'Hola Mundo'` (primera letra de cada palabra) |

`INITCAP` en PostgreSQL es el equivalente al `.title()` de pandas: capitaliza la primera letra de cada palabra (separadas por no-letras).

In [ ]:
%%sql
SELECT
    company_name                   AS original,
    UPPER(company_name)            AS upper,
    LOWER(company_name)            AS lower,
    INITCAP(LOWER(company_name))   AS initcap
FROM      northwind_dwh.dim_customer
ORDER BY  company_name
LIMIT 5;

Caso típico en ETL: **uniformar mayúsculas/minúsculas antes de comparar o agrupar**. Sin esto, `'Mexico'` y `'MEXICO'` aparecen como dos categorías distintas en un `GROUP BY country` aunque sean el mismo país.

In [ ]:
%%sql
-- Conteo agrupando con INITCAP — neutraliza variantes de capitalización
SELECT
    INITCAP(LOWER(country)) AS pais_normalizado,
    COUNT(*)                AS clientes
FROM      northwind_dwh.dim_customer
GROUP BY  INITCAP(LOWER(country))
ORDER BY  clientes DESC
LIMIT 5;

## Limpieza — `trim`, `ltrim`, `rtrim`

Eliminan caracteres del **inicio y/o final** de un string. Por default operan sobre espacios, pero puedes pasarles un segundo argumento para quitar otros caracteres:

| Función | Quita |
|---|---|
| `TRIM(s)` | Espacios al inicio y final |
| `LTRIM(s)` | Solo a la izquierda (**l**eft) |
| `RTRIM(s)` | Solo a la derecha (**r**ight) |
| `TRIM(BOTH 'x' FROM s)` | Cualquier `'x'` al inicio o final |
| `LTRIM(s, '0')` | Ceros a la izquierda |

Sintaxis menos verbosa: `TRIM('xy' FROM s)` también funciona en PostgreSQL.

In [ ]:
%%sql
SELECT
    '  hola mundo  '       AS con_espacios,
    TRIM('  hola mundo  ') AS sin_espacios,
    LTRIM('00012300', '0') AS sin_ceros_izq,
    RTRIM('00012300', '0') AS sin_ceros_der,
    TRIM('0' FROM '00012300') AS sin_ceros_ambos,
    TRIM('###título###', '#') AS sin_hashes;

**Cuidado importante:** un espacio invisible al inicio o al final **no se ve en pantalla pero rompe joins y comparaciones**. Si un `customer_id` vino con un espacio extra de un export sucio, no va a matchear con el mismo ID limpio en la otra tabla, y la fila desaparece silenciosamente del JOIN.

Por eso `TRIM` es la primera operación que aplicas a columnas string de fuentes externas — antes de cualquier validación o join.

## Búsqueda — `position`, `LIKE`, `ILIKE`

Tres herramientas para preguntar "¿este texto contiene/empieza/termina con X?":

### `POSITION(sub IN s)` — primera ocurrencia

Devuelve la **posición** (1-based) donde aparece `sub` dentro de `s`, o `0` si no aparece. `STRPOS(s, sub)` es alias con argumentos invertidos.

In [ ]:
%%sql
SELECT
    POSITION('@' IN 'usuario@ejemplo.com')   AS pos_arroba,
    POSITION('X' IN 'usuario@ejemplo.com')   AS pos_no_existe,
    STRPOS('hola mundo', 'mundo')            AS pos_strpos;

### `LIKE` y `ILIKE` — patrones con wildcards

Operadores de **matching de patrones** con dos wildcards:

| Wildcard | Significado |
|---|---|
| `%` | Cero o más caracteres cualquiera |
| `_` | Exactamente **un** caracter |

Diferencia clave:

- **`LIKE`** es **case-sensitive** (`'Mexico' LIKE 'mex%'` da `false`).
- **`ILIKE`** es **case-insensitive** — extensión de PostgreSQL (`'Mexico' ILIKE 'mex%'` da `true`).

Para datos sucios donde la capitalización no es confiable, `ILIKE` es casi siempre lo que quieres.

In [ ]:
%%sql
-- Clientes cuya empresa empieza con 'B'
SELECT customer_id, company_name, country
FROM   northwind_dwh.dim_customer
WHERE  company_name LIKE 'B%'
ORDER BY company_name
LIMIT 5;

In [ ]:
%%sql
-- ILIKE: buscar 'restaurant' sin importar capitalización en cualquier parte del nombre
SELECT customer_id, company_name
FROM   northwind_dwh.dim_customer
WHERE  company_name ILIKE '%restaurant%'
ORDER BY company_name;

**Truco común para case-insensitive sin `ILIKE`:** comparar contra la versión en minúsculas de ambos lados.

```sql
WHERE LOWER(company_name) LIKE LOWER('%restaurant%')
```

Funciona en motores que **no** tienen `ILIKE` (MySQL, SQLite, SQL Server). En PostgreSQL es más limpio usar `ILIKE` directamente.

## Extracción — `substring`, `left`, `right`, `split_part`

Para obtener pedazos específicos de un string:

| Función | Qué hace |
|---|---|
| `SUBSTRING(s FROM pos FOR n)` | A partir de `pos` (1-based), toma `n` caracteres |
| `LEFT(s, n)` | Primeros `n` caracteres |
| `RIGHT(s, n)` | Últimos `n` caracteres |
| `SPLIT_PART(s, sep, k)` | Divide por `sep` y devuelve el **k-ésimo** trozo (1-based) |

In [ ]:
%%sql
SELECT
    SUBSTRING('Hello World' FROM 1 FOR 5) AS sub_primer_palabra,
    SUBSTRING('Hello World' FROM 7 FOR 5) AS sub_segunda,
    LEFT('Hello World', 5)                AS left_5,
    RIGHT('Hello World', 5)               AS right_5,
    SPLIT_PART('ana@unam.mx', '@', 1)     AS usuario,
    SPLIT_PART('ana@unam.mx', '@', 2)     AS dominio,
    SPLIT_PART('2024-05-15', '-', 2)      AS mes;

**`SPLIT_PART` es de los más útiles** en limpieza de datos: cualquier campo con separador conocido (CSV embebido en una celda, paths de archivo, fechas en formato custom, emails) se parsea con un solo `SPLIT_PART`. Para extraer **todos** los trozos a la vez, en PostgreSQL existe `STRING_TO_ARRAY`:

```sql
STRING_TO_ARRAY('ana@unam.mx', '@')   -- {ana, unam.mx}
```

Devuelve un array, que puedes indexar (`array[1]`, `array[2]`) o expandir con `UNNEST`.

## Composición — `||`, `concat`, `format`

Tres formas de **unir strings**. La diferencia importa cuando hay NULLs:

| Operador / función | Sintaxis | Con NULL |
|---|---|---|
| `\|\|` | `a \|\| b \|\| c` | Si **cualquier** operando es NULL, el resultado es NULL |
| `CONCAT(...)` | `CONCAT(a, b, c)` | NULL se trata como string vacío `''` |
| `FORMAT(...)` | `FORMAT('%s tiene %s', n, edad)` | NULL se imprime como `'<NULL>'` |

El comportamiento de NULL en `||` es **estándar SQL**, pero suele sorprender:

In [ ]:
%%sql
SELECT
    'Hola' || ' ' || 'mundo'           AS concat_op,
    'Hola' || NULL || 'mundo'          AS concat_op_con_null,         -- → NULL
    CONCAT('Hola', NULL, 'mundo')      AS concat_func_con_null,       -- → 'Holamundo'
    FORMAT('%s tiene %s años', 'Ana', NULL) AS format_con_null;       -- → 'Ana tiene  años' (interpola '')

**Reglas operativas:**

- **`||`** cuando los operandos están garantizados no-NULL (después de un `COALESCE`, por ejemplo). Es la opción estándar SQL.
- **`CONCAT`** cuando puede haber NULLs y quieres que se ignoren (los trata como `''`).
- **`FORMAT`** para plantillas con varios `%s`, similar a `printf` de C o f-strings de Python. Más legible cuando son 3+ campos.

In [ ]:
%%sql
-- Concatenación con COALESCE para manejar NULL explícitamente
SELECT
    customer_id,
    city || ', ' || COALESCE(region || ', ', '') || country AS direccion_completa
FROM     northwind_dwh.dim_customer
ORDER BY country, city
LIMIT 8;

`COALESCE(region || ', ', '')` evalúa: si `region` es NULL, todo el sub-expresión `region || ', '` también es NULL, y `COALESCE` lo reemplaza por `''`. Resultado: la coma de la región solo aparece cuando sí hay región.

## Regex — `regexp_replace` y `regexp_match`

Cuando `REPLACE`/`SPLIT_PART` ya no alcanzan (patrones complejos, captura de grupos), PostgreSQL trae **POSIX regex**.

### `REGEXP_REPLACE(s, patron, reemplazo [, flags])`

Reemplazo basado en regex. Más potente que `REPLACE` porque acepta patrones variables.

Flag útil: `'g'` (global) — reemplaza **todas** las ocurrencias, no solo la primera.

In [ ]:
%%sql
SELECT
    REGEXP_REPLACE('(55) 1234-5678', '[^0-9]', '', 'g')      AS solo_digitos,
    REGEXP_REPLACE('Hola   mundo   espacios', '\s+', ' ', 'g') AS espacios_colapsados,
    REGEXP_REPLACE('$1,234.56', '[$,]', '', 'g')             AS sin_simbolos_moneda;

### `REGEXP_MATCH(s, patron)` — capturar grupos

Devuelve un **array** con los grupos `(...)` capturados, o `NULL` si no hay match. Útil para **extraer** valores de strings con formato fijo.

In [ ]:
%%sql
-- Extraer el número de baños de "2 baths" / "1.5 bath" / "Half-bath"
SELECT
    'Half-bath'      AS texto, (REGEXP_MATCH('Half-bath',      '(\d+(?:\.\d+)?)'))[1] AS numero
UNION ALL
SELECT '2 baths',                 (REGEXP_MATCH('2 baths',         '(\d+(?:\.\d+)?)'))[1]
UNION ALL
SELECT '1.5 baths',               (REGEXP_MATCH('1.5 baths',       '(\d+(?:\.\d+)?)'))[1]
UNION ALL
SELECT '0 shared baths',          (REGEXP_MATCH('0 shared baths',  '(\d+(?:\.\d+)?)'))[1];

Interpretación del patrón `(\d+(?:\.\d+)?)`:

- `\d+` — uno o más dígitos.
- `(?:\.\d+)?` — opcionalmente, un punto decimal seguido de más dígitos.
- `(...)` — grupo de captura (lo que `REGEXP_MATCH` devolverá).

`(?:...)` es un **grupo no-capturador**: agrupa para el cuantificador `?` pero no se incluye en el array de resultados.

> **Tip:** [regexr.com](https://regexr.com/) te deja construir y probar el patrón con muestras reales antes de incrustarlo en SQL — mismo recurso que viste en el Tema 04.

## Caso integrador — limpiar Airbnb con todo lo aprendido

Tres columnas sucias del bronze de Airbnb. Vamos a parsearlas a tipos analizables aplicando lo de este notebook.

**1. `price`** — texto con formato `'$1,234.00'`. Queremos `NUMERIC`.

In [ ]:
%%sql
SELECT
    price                                                   AS original,
    REGEXP_REPLACE(price, '[$,]', '', 'g')::NUMERIC         AS limpio
FROM     airbnb.listings
WHERE    price IS NOT NULL
LIMIT 5;

**2. `bathrooms_text`** — texto libre. Queremos el número (0 si dice `'Half-bath'`).

In [ ]:
%%sql
SELECT
    bathrooms_text                                          AS original,
    (REGEXP_MATCH(bathrooms_text, '(\d+(?:\.\d+)?)'))[1]::NUMERIC AS numero,
    bathrooms_text ILIKE '%shared%'                         AS es_compartido,
    bathrooms_text ILIKE '%half%'                           AS es_medio_bano
FROM     airbnb.listings
WHERE    bathrooms_text IS NOT NULL
LIMIT 10;

**3. `host_response_rate`** — texto como `'95%'` o `'N/A'`. Queremos `NUMERIC` entre 0 y 100, con `NULL` cuando es `'N/A'`.

In [ ]:
%%sql
SELECT
    host_response_rate                                          AS original,
    NULLIF(host_response_rate, 'N/A')                           AS sin_na,
    REPLACE(NULLIF(host_response_rate, 'N/A'), '%', '')::NUMERIC AS porcentaje
FROM     airbnb.listings
WHERE    host_response_rate IS NOT NULL
LIMIT 10;

**`NULLIF(col, valor)`** es un patrón muy útil: devuelve `NULL` si `col == valor`, y `col` en otro caso. Es la forma estándar SQL de convertir un "valor centinela" (`'N/A'`, `''`, `-1`, etc.) a `NULL` antes de seguir procesando.

Sin el `NULLIF`, el cast `'N/A'::NUMERIC` haría que la query entera fallara con un `invalid input syntax for type numeric: "N/A"`.

## Combinando todo — query analítica sobre Airbnb limpio

Con las tres columnas parseadas, ya puedes calcular agregados reales (los del Notebook 01 ahora funcionan):

In [ ]:
%%sql
-- Precio promedio y tasa de respuesta promedio por alcaldía
SELECT
    neighbourhood_cleansed                                                AS alcaldia,
    COUNT(*)                                                              AS listings,
    ROUND(AVG(REGEXP_REPLACE(price, '[$,]', '', 'g')::NUMERIC), 2)        AS precio_promedio,
    ROUND(AVG(REPLACE(NULLIF(host_response_rate, 'N/A'), '%', '')::NUMERIC), 1) AS resp_promedio_pct
FROM     airbnb.listings
WHERE    price IS NOT NULL
GROUP BY neighbourhood_cleansed
ORDER BY listings DESC
LIMIT 5;

Esa query consolidad todo: limpieza con `REPLACE`/`REGEXP_REPLACE`, manejo de centinela con `NULLIF`, cast a `NUMERIC`, agregación con `AVG`, conteo con `COUNT(*)`, `GROUP BY` por alcaldía, `ORDER BY` por la métrica de interés. Es **lo que en pandas tomaría 15 líneas** — en SQL queda en una sola query expresiva.

**Punto pedagógico de cierre:** las funciones de string en SQL son lo que hace viable cargar datos sucios al DWH como `TEXT` (bronze) y refinarlos query-a-query, en vez de pre-procesar todo en Python antes de cargar. Es el patrón que justifica el bronze layer del Tema 02.

## Cierre

Lo que cubriste:

| Tema | Funciones clave |
|---|---|
| Capitalización | `UPPER`, `LOWER`, `INITCAP` |
| Limpieza de bordes | `TRIM`, `LTRIM`, `RTRIM` (con caracteres custom) |
| Búsqueda | `POSITION`, `STRPOS`, `LIKE`, `ILIKE` |
| Extracción | `SUBSTRING`, `LEFT`, `RIGHT`, `SPLIT_PART` |
| Composición | `\|\|`, `CONCAT`, `FORMAT` (cuidado con NULL) |
| Regex | `REGEXP_REPLACE`, `REGEXP_MATCH` |
| Centinelas a NULL | `NULLIF(col, valor)` |

El siguiente notebook (**03 — Funciones de fechas**) cierra el bloque de funciones predefinidas: `date_trunc`, `extract`, `INTERVAL`, `age`, y queries temporales sobre `fact_sales` × `dim_date` que demuestran el valor del modelo dimensional para análisis por período.

---

<p align="center">
<a href="01_funciones_agregadas.ipynb">← Anterior: Notebook 01</a> | <a href="Readme.md">Volver al índice</a> | <a href="03_funciones_de_fechas.ipynb">Siguiente: Notebook 03 — Funciones de fechas →</a>
</p>